In [1]:

import json
import pennylane as qml
import pennylane.numpy as np
import scipy

H = np.array([[ 0.39488016,  0.04722628, -0.17943126, -0.03673282],
        [ 0.04722628,  0.37758558,  0.12997088,  0.17848188],
        [-0.17943126,  0.12997088,  0.53582574,  0.01115543],
        [-0.03673282,  0.17848188,  0.01115543,  0.29170851]])

U = scipy.linalg.expm(2 * np.pi * 1j * H)

def state_prep(params, wires):
    """
    Implements the state preparation circuit.
    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the circuit.
        - wires (list): Labels for the circuit wires.
    """
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[1])

dev = qml.device('lightning.qubit', wires=range(8))

@qml.qnode(dev)
def qpe_circuit(params):
    """
    Implements the QPE routine for the Unitary U, including initial state 
    preparation using the state_prep subcircuit.
    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - np.tensor(float): Computational basis probabilities in the estimation wires.
    """
    estimation_wires = list(range(6))
    target_wires = [6, 7]
    # Step 1: Hadamard on estimation wires
    for w in estimation_wires:
        qml.Hadamard(wires=w)
    # Step 2: Prepare the estimated eigenstate in target wires
    state_prep(params, wires=target_wires)
    # Step 3: QPE controlled-U^{2^j} gates
    # Use correct order: LSB (estimation_wires[5]) gets U^1, MSB (estimation_wires[0]) gets U^{32}
    for j, est_wire in enumerate(estimation_wires[::-1]):
        power = 2 ** j
        # Use exponentiation of the Hamiltonian for U^{2^j}
        CU = scipy.linalg.expm(2 * np.pi * 1j * H * power)
        qml.ControlledQubitUnitary(CU, control_wires=est_wire, wires=target_wires)
    # Step 4: Inverse QFT on estimation wires
    qml.adjoint(qml.QFT)(wires=estimation_wires)
    # Step 5: Return probabilities on estimation wires
    return qml.probs(estimation_wires)

def compute_statistics(params):
    """
    Computes the phase and its uncertainty by postprocessing the results of the QPE circuit.
    Args:
        - params (np.array(float)): Angles [theta_1, theta_2, theta_3] parametrizing
        the RY rotations in the state_prep circuit.
    Returns:
        - mu (float): The phase calculated as a weighted average.
        - sigma (float): The uncertainty calculated as the standard deviation
        for the phase.
    """
    probs = qpe_circuit(params)
    num_est = 6
    phases = np.arange(2**num_est) / 2**num_est  # phi_i = k / 64
    # Safety padding (likely not needed but robust)
    if probs.shape[0] < phases.shape[0]:
        probs = np.pad(probs, (0, phases.shape[0]-probs.shape[0]))
    mu = np.sum(probs * phases)
    sigma = np.sqrt(np.sum(probs * (phases - mu) ** 2))
    return float(np.real(mu)), float(np.real(sigma))

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = list(compute_statistics(ins))
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, atol = 1e-4)

# Public test cases
test_cases = [
    ('[1.35889209, -0.6219561, -1.31577162]', '[0.121662, 0.120539]'),
    ('[4.40084815, -0.77288063,  0.6425846]', '[0.303264, 0.043692]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")
    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")


Running test case 0 with input '[1.35889209, -0.6219561, -1.31577162]'...
Correct!
Running test case 1 with input '[4.40084815, -0.77288063,  0.6425846]'...
Correct!
